# 05 — The numbers

Five things to measure, each with error bars.

**With 40 cases, "88%" is misleading** — it sounds precise and it isn't. Report
`0.88 [0.74, 0.96]` instead.

Reads whatever notebook 04 wrote. Set `RUN_NAME` below to `"smoke"` while iterating and
`"full"` once the full experiment has been run.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

RUN_NAME = "smoke"          # <- change to "full" after running 04 with RUN_FULL = True

RESULT_DIR = Path("data/results") / RUN_NAME
runs = {p.stem: json.load(open(p)) for p in sorted(RESULT_DIR.glob("*.json"))}

if not runs:
    raise FileNotFoundError(
        f"No result files in {RESULT_DIR}. Run notebook 04 first with "
        f"RUN_FULL = {RUN_NAME == 'full'}.")

ORDER = ["row0_context_only", "row1_naive", "row2_structure",
         "row3_hybrid", "row4_rerank", "row5_full"]
AVAILABLE = [n for n in ORDER if n in runs]
MAIN = "row5_full" if "row5_full" in runs else AVAILABLE[-1]

print(f"run: {RUN_NAME}   (main config for the detailed plots: {MAIN})")
for name in AVAILABLE:
    rows = runs[name]["rows"]
    print(f"  {name:<20} {len(rows):>5} decisions   "
          f"{len({r['case_id'] for r in rows}):>2} cases   "
          f"model={runs[name].get('model')}")

## Error bars

Wilson for proportions (correct at small n, unlike the usual formula). Bootstrap for F1 —
resample the decisions 1000 times and look at the spread.

In smoke mode there are only a handful of decisions, so every interval will be nearly the
full width of the axis. That is the interval telling the truth, not a bug.

In [ ]:
import math

def wilson(successes, n, z=1.96):
    """95% interval for a proportion."""
    if n == 0:
        return (float("nan"),) * 3
    p = successes / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return p, max(0, centre - half), min(1, centre + half)

def bootstrap(items, stat, n_resamples=1000, seed=0):
    if not items:
        return (float("nan"),) * 3
    rng = np.random.default_rng(seed)
    n = len(items)
    draws = [stat([items[i] for i in rng.integers(0, n, n)]) for _ in range(n_resamples)]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return stat(items), float(lo), float(hi)

def fmt(point, lo, hi):
    if any(math.isnan(v) for v in (point, lo, hi)):
        return "—"
    return f"{point:.2f} [{lo:.2f}, {hi:.2f}]"

print(fmt(*wilson(35, 40)))

## The five measurements

Field names come straight from the rows notebook 04 writes:

| row field | what it is |
|---|---|
| `gold` | the frozen oracle's label |
| `predicted_raw` | what the model said |
| `predicted_final` | after abstention, when the config enables it |
| `quote_status` | `supported` / `close` / `wrong_doc` / `made_up` / `empty`, by string match |
| `retrieval_rank` | 1-based rank of the criterion's passage, `None` if never retrieved |

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

LABELS  = ["met", "unmet", "insufficient_evidence"]
ABSTAIN = "insufficient_evidence"

def macro_f1(rows, field="predicted_final"):
    return f1_score([r["gold"] for r in rows], [r[field] for r in rows],
                    labels=LABELS, average="macro", zero_division=0)

def summarise(run):
    rows = run["rows"]
    n    = len(rows)
    cfg  = run["config"]

    # Row 0 puts the whole of L33718 into one pseudo-chunk, so every criterion "ranks"
    # first by construction. That is not a retrieval result, so it is not reported.
    if cfg["retrieval"] is None:
        found = "—"
    else:
        found = fmt(*wilson(sum(r["retrieval_rank"] is not None for r in rows), n))

    answered = [r for r in rows if r["predicted_final"] != ABSTAIN]
    return [
        found,
        fmt(*bootstrap(rows, macro_f1)),
        fmt(*wilson(sum(r["quote_status"] == "supported" for r in rows), n)),
        fmt(*wilson(sum(r["quote_status"] == "made_up"   for r in rows), n)),
        fmt(*wilson(sum(r["predicted_final"] == ABSTAIN  for r in rows), n)),
        fmt(*wilson(sum(r["gold"] == r["predicted_final"] for r in answered), len(answered))),
    ]

## The table

In [ ]:
HEAD = ["config", "found rule", "F1", "quotes real", "made up", "abstained", "acc. answered"]
WIDTH = [20, 20, 20, 20, 20, 20, 20]

print(" | ".join(h.ljust(w) for h, w in zip(HEAD, WIDTH)))
print("-" * (sum(WIDTH) + 3 * len(WIDTH)))
for name in AVAILABLE:
    cells = [name] + summarise(runs[name])
    print(" | ".join(str(c).ljust(w) for c, w in zip(cells, WIDTH)))

# then paste this into README.md

## What abstention actually changed

`predicted_raw` is what the model said; `predicted_final` is what survived quote
verification. Only configs with `abstain: True` differ between the two.

In [ ]:
print(f"{'config':<20} {'raw F1':>8} {'final F1':>9} {'forced abstentions':>20}")
for name in AVAILABLE:
    rows = runs[name]["rows"]
    forced = sum(r["predicted_raw"] != ABSTAIN and r["predicted_final"] == ABSTAIN
                 for r in rows)
    print(f"{name:<20} {macro_f1(rows, 'predicted_raw'):>8.2f} "
          f"{macro_f1(rows, 'predicted_final'):>9.2f} {forced:>20}")

## Confusion matrix

In [ ]:
rows = runs[MAIN]["rows"]
cm = confusion_matrix([r["gold"] for r in rows],
                      [r["predicted_final"] for r in rows], labels=LABELS)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_xticks(range(3), LABELS, rotation=45, ha="right")
ax.set_yticks(range(3), LABELS)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title(MAIN)
plt.tight_layout()

## Risk–coverage curve

Sweep a threshold: how much it answers, against how wrong it is on what it answered. The
shape of this curve is the entire argument for letting it abstain.

**Two signals could order the decisions, and they are not equally trustworthy.**

1. **`quote_status` — objective.** Produced by string matching against the policy text,
   not by the model. This is the primary curve and the one to quote.
2. **`model_reported_confidence` — self-reported and uncalibrated.** An LLM's stated
   confidence is not a probability; it is generated text. It is plotted only as an
   exploratory comparison. Do not present it as a reliability estimate, and do not use it
   to decide anything. If the two curves disagree, the objective one is the finding.

The curve uses `predicted_raw`, because the question it asks is whether a signal can
separate the model's right answers from its wrong ones *before* abstention is applied.

In [ ]:
# higher = more trustworthy
VERIFICATION_ORDER = {"supported": 3, "close": 2, "wrong_doc": 1, "made_up": 0, "empty": 0}

def risk_coverage(rows, key):
    scored  = sorted(rows, key=key, reverse=True)
    correct = np.array([r["gold"] == r["predicted_raw"] for r in scored], dtype=float)
    k       = np.arange(1, len(scored) + 1)
    return k / len(scored), 1 - np.cumsum(correct) / k

rows = runs[MAIN]["rows"]
fig, ax = plt.subplots(figsize=(6, 4))

cov, err = risk_coverage(rows, lambda r: VERIFICATION_ORDER.get(r["quote_status"], 0))
ax.plot(cov, err, label="quote verification (objective)", linewidth=2)

# Exploratory only -- see the note above. Absent if the run predates the schema change.
conf = [r.get("model_reported_confidence") for r in rows]
if any(c is not None for c in conf):
    cov2, err2 = risk_coverage(rows, lambda r: (r.get("model_reported_confidence") or 0.0))
    ax.plot(cov2, err2, "--", label="model self-reported confidence (uncalibrated)")

ax.set_xlabel("coverage (fraction answered)")
ax.set_ylabel("error rate on answered")
ax.set_title(f"risk–coverage, {MAIN}")
ax.legend()
plt.tight_layout()

## Diagnostics

Where the errors actually live. With `n = 40` the headline table hides almost everything,
so this is usually the more useful cell.

In [ ]:
rows = runs[MAIN]["rows"]

omitted = sum(r["model_omitted"] for r in rows)
print(f"criteria the model was asked for but did not return: {omitted}/{len(rows)}")

missed = sum(r["retrieval_rank"] is None for r in rows)
print(f"criterion passage never retrieved:                   {missed}/{len(rows)}")
print(f"distinct chunks retrieved across all cases:          "
      f"{len({c for r in rows for c in r['retrieved_chunk_ids']})}")

print("\nretrieved documents (anything other than L33718 is a decoy):")
for doc, k in Counter(d for r in rows for d in r["retrieved_doc_ids"]).most_common():
    print(f"  {doc:<12} {k:>5}{'' if doc == 'L33718' else '   <- decoy'}")

print("\nmacro-F1 by bucket:")
for b in sorted({r["bucket"] for r in rows}):
    sub = [r for r in rows if r["bucket"] == b]
    print(f"  {b:<14} n={len(sub):<4} {macro_f1(sub):.2f}")

print("\naccuracy by criterion:")
for cid in sorted({r["criterion_id"] for r in rows}):
    sub = [r for r in rows if r["criterion_id"] == cid]
    ok  = sum(r["gold"] == r["predicted_final"] for r in sub)
    print(f"  {cid:<22} {ok}/{len(sub)}")

## Handwritten vs templated

If the model does much worse on cases written by hand, the templates were too easy and the
README has to say so.

In [ ]:
rows = runs[MAIN]["rows"]
hand = [r for r in rows if r["hand_written"]]
tmpl = [r for r in rows if not r["hand_written"]]

if not hand:
    print(f"No handwritten cases: all {len(tmpl)} decisions come from generated text.")
    print("Every field is phrased from a small bank, so a model could be matching phrasing")
    print("rather than reading the record, and nothing here would reveal it.")
    print("Either rewrite some cases by hand in notebook 02 (set hand_written = True),")
    print("or state this in the README limitations. Do not quote these numbers without it.")
else:
    print("handwritten:", fmt(*bootstrap(hand, macro_f1)), f"(n={len(hand)})")
    print("templated:  ", fmt(*bootstrap(tmpl, macro_f1)), f"(n={len(tmpl)})")